In [1]:
# ================================================================
# COMPLETE COLAB SCRIPT: BioHub CIF → Clean PDB + Metadata
# ================================================================

# -----------------------------------------------------------------
# 1. Install required packages (run once)
# -----------------------------------------------------------------
!pip install biopython pandas numpy openpyxl xlsxwriter -q

# -----------------------------------------------------------------
# 2. Imports
# -----------------------------------------------------------------
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser, PDBIO, Select
from Bio.PDB.Polypeptide import PPBuilder
from google.colab import files

# -----------------------------------------------------------------
# 3. Helper classes and functions
# -----------------------------------------------------------------

class CleanSelect(Select):
    """Select all atoms but write only ATOM/HETATM/TER lines."""
    def accept_atom(self, atom):
        return True

def clean_pdb_file(pdb_path):
    """
    Remove all lines except ATOM, HETATM, and TER.
    This ensures strict PDB format expected by Vina.
    """
    with open(pdb_path, 'r') as f:
        lines = f.readlines()
    clean = [line for line in lines if line.startswith(('ATOM', 'HETATM', 'TER'))]
    with open(pdb_path, 'w') as f:
        f.writelines(clean)

def get_sequence(structure):
    """Extract protein sequence from structure."""
    ppb = PPBuilder()
    sequences = []
    for pp in ppb.build_peptides(structure):
        sequences.append(str(pp.get_sequence()))
    return "".join(sequences)

def get_plddt_stats(structure):
    """Extract mean, min, max pLDDT from B‑factor column."""
    b_factors = [atom.get_bfactor() for atom in structure.get_atoms()]
    if not b_factors:
        return np.nan, np.nan, np.nan
    return np.mean(b_factors), np.min(b_factors), np.max(b_factors)

# -----------------------------------------------------------------
# 4. Upload CIF files
# -----------------------------------------------------------------
print("📤 Please upload your CIF files (from BioHub/ESMFold).")
uploaded = files.upload()

# Create directories
cif_dir = '/content/cif_files/'
pdb_dir = '/content/pdb_clean/'
os.makedirs(cif_dir, exist_ok=True)
os.makedirs(pdb_dir, exist_ok=True)

# Save uploaded files to cif_dir
for filename, content in uploaded.items():
    with open(os.path.join(cif_dir, filename), 'wb') as f:
        f.write(content)
print(f"✅ Saved {len(uploaded)} CIF files to {cif_dir}")

# -----------------------------------------------------------------
# 5. Convert CIF → Clean PDB
# -----------------------------------------------------------------
cif_files = glob.glob(os.path.join(cif_dir, '*.cif'))
if not cif_files:
    raise SystemExit("No CIF files found. Please upload .cif files.")

print(f"\n🔄 Converting {len(cif_files)} CIF files to clean PDB...")

parser = MMCIFParser(QUIET=True)
io = PDBIO()
metadata = []

for cif_path in cif_files:
    base = os.path.splitext(os.path.basename(cif_path))[0]
    pdb_path = os.path.join(pdb_dir, base + '.pdb')

    try:
        structure = parser.get_structure(base, cif_path)

        # Write PDB (only ATOM/HETATM/TER)
        io.set_structure(structure)
        io.save(pdb_path, CleanSelect())
        clean_pdb_file(pdb_path)  # extra safeguard

        # Extract metadata
        seq = get_sequence(structure)
        mean_plddt, min_plddt, max_plddt = get_plddt_stats(structure)

        metadata.append({
            'protein_id': base,
            'source_organism': 'Homo sapiens',  # adjust if needed
            'sequence_length': len(seq),
            'mean_plddt': mean_plddt,
            'min_plddt': min_plddt,
            'max_plddt': max_plddt,
            'coverage': '100%',  # assume full-length
            'has_dbd': 'Yes' if len(seq) >= 292 else 'No',  # rough check
            'dbd_residues': '102-292' if len(seq) >= 292 else 'Not Available',
            'has_tetramerization': 'Yes' if len(seq) >= 393 else 'No',
            'rmsd_to_reference': 'Not Available',
            'cluster_rep_hash': 'Not Available',
            'notes': '',
            'file_name': base + '.pdb',
            'sequence': seq,
        })

        print(f"✅ Converted {base}")

    except Exception as e:
        print(f"❌ Error converting {base}: {e}")

# -----------------------------------------------------------------
# 6. Create metadata DataFrame
# -----------------------------------------------------------------
df = pd.DataFrame(metadata)

# Sort by mean pLDDT (best first)
df = df.sort_values('mean_plddt', ascending=False).reset_index(drop=True)

# Add a rank column
df.insert(0, 'rank', range(1, len(df)+1))

# Save CSV and Excel
csv_path = '/content/structure_metadata.csv'
excel_path = '/content/structure_metadata.xlsx'

df.to_csv(csv_path, index=False, float_format='%.10f')
print(f"\n📊 Metadata CSV saved to {csv_path}")

# Create Excel with two sheets
with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='All Results', index=False, float_format='%.10f')

    # Summary statistics
    summary = {
        'Metric': [
            'Total Proteins Analyzed',
            'Mean Sequence Length',
            'Median Sequence Length',
            'Min Sequence Length',
            'Max Sequence Length',
            'Mean pLDDT (Overall)',
            'Median pLDDT (Overall)',
            'Min pLDDT (Overall)',
            'Max pLDDT (Overall)',
            'Number with DNA-binding Domain (Assumed)',
            'Number with Tetramerization Domain (Assumed)',
            'RMSD Summary',
            'Data Quality Warning: Missing RMSD values',
            'Data Quality Warning: Missing Cluster Representative Hash'
        ],
        'Value': [
            len(df),
            df['sequence_length'].mean(),
            df['sequence_length'].median(),
            df['sequence_length'].min(),
            df['sequence_length'].max(),
            df['mean_plddt'].mean(),
            df['mean_plddt'].median(),
            df['min_plddt'].min(),
            df['max_plddt'].max(),
            f"{df[df['has_dbd']=='Yes'].shape[0]} ({df[df['has_dbd']=='Yes'].shape[0]/len(df)*100:.2f}%)",
            f"{df[df['has_tetramerization']=='Yes'].shape[0]} ({df[df['has_tetramerization']=='Yes'].shape[0]/len(df)*100:.2f}%)",
            'Not Applicable (no reference structure provided)',
            f"{df[df['rmsd_to_reference']=='Not Available'].shape[0]} proteins",
            f"{df[df['cluster_rep_hash']=='Not Available'].shape[0]} proteins"
        ]
    }
    summary_df = pd.DataFrame(summary)
    summary_df.to_excel(writer, sheet_name='Summary Statistics', index=False)

print(f"📊 Excel report saved to {excel_path}")

# -----------------------------------------------------------------
# 7. Zip all clean PDB files
# -----------------------------------------------------------------
zip_path = '/content/clean_pdb_files.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for pdb in glob.glob(os.path.join(pdb_dir, '*.pdb')):
        zipf.write(pdb, os.path.basename(pdb))
print(f"📦 PDB files zipped to {zip_path}")

# -----------------------------------------------------------------
# 8. Download files
# -----------------------------------------------------------------
print("\n⬇️ Downloading your clean PDB zip and Excel report...")
files.download(zip_path)
files.download(excel_path)

print("\n🎉 All done! You now have:")
print(f"  - Clean PDB files in {pdb_dir}")
print(f"  - Metadata CSV: {csv_path}")
print(f"  - Excel report: {excel_path}")
print(f"  - ZIP archive: {zip_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 17.2 MB/s eta 0:00:00
📤 Please upload your CIF files (from BioHub/ESMFold).


Saving TP53_A159V.cif to TP53_A159V.cif
Saving TP53_A161T.cif to TP53_A161T.cif
Saving TP53_C17F.cif to TP53_C17F.cif
Saving TP53_C44F.cif to TP53_C44F.cif
Saving TP53_C102Y.cif to TP53_C102Y.cif
Saving TP53_C135F.cif to TP53_C135F.cif
Saving TP53_C137F.cif to TP53_C137F.cif
Saving TP53_C137Y.cif to TP53_C137Y.cif
Saving TP53_C141Y.cif to TP53_C141Y.cif
Saving TP53_C176F.cif to TP53_C176F.cif
Saving TP53_C176Y.cif to TP53_C176Y.cif
Saving TP53_C199Y.cif to TP53_C199Y.cif
Saving TP53_C238F.cif to TP53_C238F.cif
Saving TP53_C238Y.cif to TP53_C238Y.cif
Saving TP53_C242F.cif to TP53_C242F.cif
Saving TP53_C275Y.cif to TP53_C275Y.cif
Saving TP53_E126K.cif to TP53_E126K.cif
Saving TP53_E153K.cif to TP53_E153K.cif
Saving TP53_E246K.cif to TP53_E246K.cif
Saving TP53_E247K.cif to TP53_E247K.cif
Saving TP53_E285K.cif to TP53_E285K.cif
Saving TP53_E286K.cif to TP53_E286K.cif
Saving TP53_G86S.cif to TP53_G86S.cif
Saving TP53_G113S.cif to TP53_G113S.cif
Saving TP53_G206D.cif to TP53_G206D.cif
Saving

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 All done! You now have:
  - Clean PDB files in /content/pdb_clean/
  - Metadata CSV: /content/structure_metadata.csv
  - Excel report: /content/structure_metadata.xlsx
  - ZIP archive: /content/clean_pdb_files.zip
